# **📊 Clean & Jerk Dataset — Summary Explorer**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import DEFAULT_CLEAN_DATASET_DIR, load_summary_tables, load_set_h5

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
print(f"📂 Dataset folder: {dataset_dir}")

### 📁 Load All Root Summary Tables

In [ ]:
# @title 📁 Load Summary Tables
tables = load_summary_tables(dataset_dir)
for name, df in tables.items():
    print(f"  • {name}: {len(df)} rows × {len(df.columns)} cols")

### 👥 Participants Summary (`participant_summary.csv`)

In [ ]:
# @title 👥 Participants Summary
df_participants = tables["participants"]
display_table(df_participants, clean_columns=False)

### 🏋️ Sets Summary (`sets_summary.csv`)

In [ ]:
# @title 🏋️ Sets Summary
df_sets = tables["sets"]
display_table(df_sets, clean_columns=False)

### 📡 26-IMU Device Summary (`device_summary.csv`)

In [ ]:
# @title 📡 Device Summary
df_devices = tables["devices"]
display_table(df_devices, clean_columns=False)

### ⚠️ Fault Log (`fault_log.csv`)

In [ ]:
# @title ⚠️ Fault Log
df_faults = tables["faults"]
display_table(df_faults, clean_columns=False)

### ⏱️ Phase Markers Summary (`phase_markers_summary.csv`)

In [ ]:
# @title ⏱️ Phase Markers Summary
df_markers = tables["phase_markers"]
display_table(df_markers, clean_columns=False)

### 📖 Data Dictionary (`data_dictionary.csv`)

In [ ]:
# @title 📖 Data Dictionary
df_dictionary = tables["data_dictionary"]
display_table(df_dictionary, clean_columns=False)

### 📈 Cohort & Recording Overview Plots

In [ ]:
# @title 📈 Cohort & Recording Overview
from src.explorer import plot_cohort_overview

fig_cohort = plot_cohort_overview(tables["participants"], tables["sets"])
fig_cohort.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### 🩺 Sensor Completeness Overview

In [ ]:
# @title 🩺 Sensor Completeness Overview
from src.explorer import plot_sensor_health_overview

fig_health = plot_sensor_health_overview(tables["sets"])
fig_health.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### 🎯 Inter-Annotator Agreement by Phase Marker

In [ ]:
# @title 🎯 Inter-Annotator Agreement
from src.explorer import plot_marker_agreement_summary

fig_agreement = plot_marker_agreement_summary(tables["phase_markers"])
fig_agreement.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})